In [1]:
%%writefile flights.csv
booking_id,passenger_name,from_city,to_city,airline,ticket_price,travel_class,status
1001,Aarav Mehta,Hyderabad,Delhi,IndiGo,6500,Economy,Confirmed
1002,Sana Khan,Bangalore,Mumbai,Vistara,8200,Economy,Confirmed
1003,John Mathew,Chennai,Delhi,Air India,12000,Business,Confirmed
1004,Ayesha Begum,Hyderabad,Dubai,Emirates,28000,Economy,Confirmed
1005,Vikram Rao,Mumbai,Singapore,Singapore Airlines,35000,Business,Pending
1006,Divya Sharma,Delhi,Hyderabad,IndiGo,5900,Economy,Cancelled
1007,Imran Ali,Pune,Bangalore,Akasa Air,4800,Economy,Confirmed
1008,Meera Nair,Kochi,Dubai,Emirates,26000,Economy,Confirmed
1009,Rohan Das,Kolkata,Delhi,Air India,7400,Economy,Pending
1010,Nisha Reddy,Bangalore,London,British Airways,62000,Business,Confirmed

Overwriting flights.csv


In [ ]:
j

c:\Users\cibir\AppData\Local\Programs\Python\Python312\python.exe


ModuleNotFoundError: No module named 'pyspark'

In [ ]:
df = spark.read.csv("flights.csv",header=True,inferSchema=True)
df.show()
df.printSchema()

In [ ]:
from pyspark.sql.functions import col, sum, avg, max, min, when, array_contains

df.select("passenger_name","from_city","to_city").show()

In [ ]:
df.filter(col("status")=="Confirmed").show()

In [ ]:
df.filter(col("status")=="Cancelled").show()

In [ ]:
df.filter(col("status")=="Pending").show()

In [ ]:
df.filter(col("ticket_price")>20000).show()

In [ ]:
df.filter(col("to_city").isin("Dubai","Singapore","London")).show()

In [ ]:
df.count()

In [ ]:
df.groupBy("airline").count().show()

In [ ]:
df.groupBy("status").count().show()

In [ ]:
df.filter(col("status")=="Confirmed").agg(sum("ticket_price").alias("total_revenue")).show()

In [ ]:
df.groupBy("airline").agg(avg("ticket_price").alias("average_price")).show()

In [ ]:
df.agg(max("ticket_price").alias("highest_price")).show()

In [ ]:
df.agg(min("ticket_price").alias("lowest_price")).show()

In [ ]:
df.orderBy(col("ticket_price").desc()).show()

In [ ]:
df=df.withColumn("tax",col("ticket_price")*0.05)
df.show()

In [ ]:
df=df.withColumn("final_price",col("ticket_price")+col("tax"))
df.show()

In [ ]:
df=df.withColumn("price_category",
                 when(col("ticket_price")>=30000,"Premium")
                 .when(col("ticket_price")>=10000,"Standard")
                 .otherwise("Budget"))

df.show()

In [ ]:
df.groupBy("price_category").count().show()

In [ ]:
df.filter(col("status")=="Confirmed").write.mode("overwrite").parquet("confirmed_flights.parquet")

json

In [ ]:
%%writefile hotels.json
[
{
"hotel_id":201,
"hotel_name":"Pearl Grand",
"city":"Hyderabad",
"category":"Business",
"rating":4.4,
"rooms_available":25,
"price_per_night":4500,
"amenities":["wifi","breakfast","gym"],
"contact":{
"phone":"9876500011",
"email":"pearlgrand@mail.com"
}
},
{
"hotel_id":202,
"hotel_name":"Marina Bay Stay",
"city":"Dubai",
"category":"Luxury",
"rating":4.8,
"rooms_available":12,
"price_per_night":18000,
"amenities":["wifi","pool","spa","sea_view"],
"contact":{
"phone":"9876500012",
"email":"marinabay@mail.com"
}
},
{
"hotel_id":203,
"hotel_name":"Budget Inn",
"city":"Delhi",
"category":"Budget",
"rating":3.9,
"rooms_available":40,
"price_per_night":2200,
"amenities":["wifi"],
"contact":{
"phone":null,
"email":"budgetinn@mail.com"
}
},
{
"hotel_id":204,
"hotel_name":"Hill View Resort",
"city":"Kochi",
"category":"Resort",
"rating":4.5,
"rooms_available":18,
"price_per_night":7500,
"amenities":["wifi","breakfast","pool"],
"contact":{
"phone":"9876500014",
"email":null
}
},
{
"hotel_id":205,
"hotel_name":"Skyline Suites",
"city":"London",
"category":"Luxury",
"rating":4.7,
"rooms_available":8,
"price_per_night":22000,
"amenities":["wifi","breakfast","spa"],
"contact":{
"phone":"9876500015",
"email":"skyline@mail.com"
}
}
]

In [ ]:
hdf=spark.read.option("multiline","true").json("hotels.json")
hdf.show(truncate=False)
hdf.printSchema()

In [ ]:
hdf.select("hotel_name","city","rating").show()

In [ ]:
hdf.filter(col("category")=="Luxury").show(truncate=False)

In [ ]:
hdf.filter(col("rating")>4.5).show(truncate=False)

In [ ]:
hdf.filter(col("rooms_available")>15).show(truncate=False)

In [ ]:
hdf.filter(col("price_per_night")>10000).show(truncate=False)

In [ ]:
hdf.filter(col("city").isin("Dubai","London")).show(truncate=False)

In [ ]:
hdf.filter(col("contact.phone").isNull()).show(truncate=False)

In [ ]:
hdf.filter(col("contact.email").isNull()).show(truncate=False)

In [ ]:
hdf.select("hotel_name","contact.phone","contact.email").show(truncate=False)

In [ ]:
hdf.filter(array_contains(col("amenities"),"wifi")).show(truncate=False)

In [ ]:
hdf.filter(array_contains(col("amenities"),"spa")).show(truncate=False)

In [ ]:
hdf.groupBy("city").count().show()

In [ ]:
hdf.groupBy("category").count().show()

In [ ]:
hdf.groupBy("category").agg(avg("rating").alias("average_rating")).show()

In [ ]:
hdf.groupBy("city").agg(avg("price_per_night").alias("average_price")).show()

In [ ]:
hdf.agg(max("price_per_night").alias("highest_price")).show()

In [ ]:
hdf=hdf.withColumn("total_potential_revenue",
                   col("rooms_available")*col("price_per_night"))

hdf.show(truncate=False)

In [ ]:
hdf.orderBy(col("rating").desc()).show(truncate=False)

In [ ]:
hotels_flattened=hdf.select(
    "hotel_id",
    "hotel_name",
    "city",
    "category",
    "rating",
    "rooms_available",
    "price_per_night",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

hotels_flattened.show(truncate=False)

In [ ]:
hotels_flattened.write.mode("overwrite").parquet("hotels_flattened.parquet")